In [6]:
%pip install -q langchain langchain-community langchain-openai faiss-cpu pymupdf
%pip install -q fastapi uvicorn requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from dotenv import load_dotenv

load_dotenv()

True

# Load

In [13]:
from langchain_community.document_loaders import PyMuPDFLoader

FILE_PATH = "./data/생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf"

loadar = PyMuPDFLoader(FILE_PATH)
docs = loadar.load()

print(len(docs))                  
print(docs[1].page_content[:200]) 

30
08
생성형 AI윤리 가이드북
PART
생성형 AI란?
01
생성형 AI가 무엇일까요?
생성형 AI(Generative AI)란, 대규모 데이터와 패턴을 학습하고 기존의 데이터를 활용하여 이용자의 요구에 
따라 텍스트, 이미지, 비디오, 음악, 코딩 등 새로운 결과를 만들어 내는 인공지능 기술입니다. 생성형 AI는 
누구나 쉽게 활용할 수 있도록 설계되었고


# Text Split

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

split_docs = text_splitter.split_documents(docs)

print(len(split_docs))

84


# Embedding & Store

In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embeddings)

# Retriever

In [16]:
retriever = vectorstore.as_retriever(search_kwargs={"k" : 4})

found = retriever.invoke("미국 콜로라도주 미술대회에서 AI가 만든 작품이 1등을 차지한 사례에서 발생한 문제의 원인과 그 의미를 설명하세요.")
for i, d in enumerate(found, 1):
    print(i, d.metadata.get("page"), d.page_content[:60].replace("\n", " "))

1 12 33 관련사례 ▶ 美 미술 공모전서 AI가 그린 그림이 1등상 받아 논란 “예술의 죽음” vs “AI도 사
2 12 완성해 제출하는 것이 좋습니다. 또한 출처를 달아 생성형 AI를 답안 작성에 어떻게 활용했는지를  교수자가 
3 12 마라톤에 참가한 것”, “클릭 몇 번으로 만든 디지털 아트”, “상을 반납하고 공개 사과하라”라는 비난이  
4 9 30 책임성 생성형 AI윤리 가이드북 PART 03 생성형 AI가 만든 작품이 미술대회 1등?  8) 202


# Prompt & LLM

In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved [context] to answer the [question]. 
If you don't know the [answer] in [context], just say that you don't know the [answer]. 
Answer in Korean.

[context]
{context}

[question]
{question}

[answer]"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Chain

In [21]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs,
    "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
question = "미국 콜로라도주 미술대회에서 AI가 만든 작품이 1등을 차지한 사례에서 발생한 문제의 원인과 그 의미를 설명하세요."
answer = rag_chain.invoke(question)

print(answer)

미국 콜로라도주 미술대회에서 AI가 만든 작품이 1등을 차지한 사례에서 발생한 문제의 원인은 수상자인 제이슨 M. 앨런이 출품한 작품이 AI 프로그램인 '미드저니'를 통해 제작되었음에도 불구하고, 일부 심사위원들이 이를 인지하지 못하고 1등상을 수여했기 때문입니다. 이로 인해 다른 예술가들은 앨런이 부정 행위를 했다고 주장하며 반발했습니다. 앨런은 미드저니 사용 사실을 미리 고지했다고 주장했지만, 이 사건은 생성형 AI의 활용에 대한 규정과 윤리에 대한 논란을 불러일으켰습니다.

이 사건의 의미는 생성형 AI의 사용이 예술 창작에 미치는 영향과 그에 대한 사회적 인식이 아직 충분히 정립되지 않았음을 보여줍니다. 또한, AI가 만든 작품이 인간의 창작물과 동일하게 평가받는 것이 적절한지에 대한 논의가 필요하다는 점을 시사합니다. 이는 예술의 정의와 창작의 본질에 대한 깊은 질문을 던지며, 향후 예술 분야에서 AI의 역할과 규제에 대한 논의가 더욱 활발해질 필요성을 강조합니다.


In [25]:
question = "생성형 AI 사용 시 개인정보 유출 위험과 이를 방지하기 위한 기업 및 개인의 윤리적 대응 방안을 비교하여 설명하세요."
answer = rag_chain.invoke(question)

print(answer)

생성형 AI 사용 시 개인정보 유출 위험은 여러 가지가 있습니다. 생성형 AI는 사용자와의 대화를 통해 데이터를 수집하고 학습하는데, 이 과정에서 사용자가 자신의 개인정보나 회사 기밀을 무의식적으로 누설할 수 있습니다. 또한, 생성형 AI는 대규모 데이터를 수집하면서 의도하지 않게 개인정보를 학습할 수 있으며, 이 데이터는 외부로 유출될 위험이 존재합니다. 실제로 챗GPT와 같은 서비스에서 대화 기록이 유출된 사례도 있었습니다.

이러한 위험을 방지하기 위해 기업과 개인은 다음과 같은 윤리적 대응 방안을 마련해야 합니다. 

1. **기업의 대응 방안**:
   - 기업들은 생성형 AI의 사용을 무조건 제한하기보다는 효과적으로 활용할 수 있는 방안을 모색하고 있습니다. 이를 위해 업무 특성을 고려한 생성형 AI 활용 방침을 제정하여 핵심 정보 유출 가능성을 차단하려고 합니다.
   - 또한, 산업통상자원부 국가기술표준원은 AI 윤리 점검 서식을 제정하여 기업들이 AI 기술을 활용할 때 윤리적 고려를 할 수 있도록 지원하고 있습니다.

2. **개인의 대응 방안**:
   - 개인 사용자들은 생성형 AI에 입력하는 정보에 대해 신중해야 하며, 개인정보나 회사 기밀과 같은 민감한 정보는 입력하지 않아야 합니다.
   - 사용자들은 기본적인 가이드라인을 준수하여 생성형 AI가 개인정보를 침해하거나 인격권을 훼손하지 않도록 노력해야 합니다.

결론적으로, 생성형 AI 사용 시 개인정보 유출 위험을 줄이기 위해서는 기업과 개인 모두가 윤리적 대응 방안을 마련하고 실천하는 것이 중요합니다.


In [26]:
question = "생성형 AI가 만들어 낸 콘텐츠가 현행 저작권법에서 보호받지 못하는 이유는 무엇인가요?"
answer = rag_chain.invoke(question)

print(answer)

생성형 AI가 만들어 낸 콘텐츠는 현행 저작권법상 저작권이 인정되지 않기 때문입니다. 저작권법에 따르면, 저작물은 인간의 사상 또는 감정을 표현한 창작물이어야 하며, 생성형 AI는 인간의 정신적 활동을 전제로 하지 않기 때문에 저작자로 인정받지 못합니다. 따라서 생성형 AI가 자동적으로 생성한 결과물은 저작권법의 보호 대상이 아닙니다.


# FastAPI & Streamlit

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Question(BaseModel):
    question: str

# http://127.0.0.1:8000/ask
# @app.post("/ask")
def ask(q: Question):
    answer = rag_chain.invoke(q.question)
    return {{"question": q.question, "answer": answer}}

In [ ]:
import requests

res = requests.post(
    # "http://127.0.0.1:8000/ask",
    json={"question": "미국 콜로라도주 미술대회에서 AI가 만든 작품이 1등을 차지한 사례에서 발생한 문제의 원인과 그 의미를 설명하세요."},
)

print(res.status_code)
print(res.json())

# RAGAS

In [ ]:
# %pip install -qU ragas "langchain-community<0.4" pandas

In [ ]:
import pandas as pd

# DATASET = "./data/ragas_testset_temp.csv"

df = pd.read_csv(DATASET_PATH)
print(df.columns.tolist())
df.head()